# CymbalGoal — Provisioning Prototype

**This is a MEASUREMENT prototype, not lab content.** It exists to answer two questions that decide
whether the startup VM can be deleted:

1. How long does a VPC-attached Colab Enterprise runtime take to cold start?
2. How long does the full load take from a notebook, versus the **2.5 min** measured from a VM?

Every phase is timed. The teaching version of this notebook — the one students see — gets written
in the Lab 1 session, not here.

**Prerequisite:** this runtime must be attached to `cymbalgoal-network`, or it cannot reach the
cluster's private IP.

In [ ]:
# ---- Config -----------------------------------------------------------------
PROJECT   = "qwiklabs-gcp-04-23ae51410344"
REGION    = "us-central1"
DB_HOST   = "10.188.244.2"          # google_alloydb_instance.primary.ip_address
DB_NAME   = "cymbalgoal"
GCS       = "gs://class-demo/alloydb-labs/cymbalgoal"

# IAM auth: username is your lab account, password is a short-lived access token.
# No stored password, and nothing sensitive in notebook output.
import subprocess
DB_USER = subprocess.run(["gcloud","config","get-value","account"],
                         capture_output=True, text=True).stdout.strip()
print("connecting as:", DB_USER)

In [ ]:
!pip install -q "psycopg[binary]" > /dev/null 2>&1
import psycopg, subprocess, time, gzip, io, json, textwrap
from contextlib import contextmanager

TIMINGS = {}

@contextmanager
def phase(name):
    t0 = time.time()
    print(f"--- {name} ---")
    yield
    TIMINGS[name] = round(time.time() - t0, 1)
    print(f"    {TIMINGS[name]} s\n")

def token():
    return subprocess.run(["gcloud","auth","print-access-token"],
                          capture_output=True, text=True).stdout.strip()

def connect(db="postgres"):
    # autocommit because CREATE DATABASE cannot run inside a transaction
    return psycopg.connect(host=DB_HOST, user=DB_USER, password=token(),
                           dbname=db, sslmode="require", autocommit=True)

def show_notices(conn):
    # This is why a notebook may beat AlloyDB Studio: the BM25 index build
    # reports k1, b, document count and average length via NOTICE, and that is
    # free instrumentation for Task 3 -- IF the surface actually shows it.
    for n in conn.notices:
        print("   ", n.strip())
    conn.notices.clear()

## Reachability

If this fails, the runtime is not VPC-attached. Nothing below can work until it is.

In [ ]:
with phase("00_connect"):
    with connect() as c:
        print(c.execute("SELECT version()").fetchone()[0])
        print(c.execute("SELECT current_user, "
                        "pg_has_role(current_user,'alloydbsuperuser','member')").fetchone())

## Phase 1 — database and extensions

Everything the startup VM did in steps 1–3. If a notebook can do this, the VM has no unique job.

In [ ]:
with phase("01_database"):
    with connect() as c:
        c.execute(f"DROP DATABASE IF EXISTS {DB_NAME} WITH (FORCE)")
        c.execute(f"CREATE DATABASE {DB_NAME}")
    print("    created", DB_NAME)

with phase("02_extensions"):
    with connect(DB_NAME) as c:
        for ext in ["vector","alloydb_scann","google_ml_integration","pg_textsearch"]:
            c.execute(f"CREATE EXTENSION IF NOT EXISTS {ext}")
        for r in c.execute("SELECT extname, extversion FROM pg_extension ORDER BY 1").fetchall():
            print("   ", r[0], r[1])

## Phase 2 — schema

`schema.sql` also creates the two ScaNN indexes, on empty tables. We drop them by name and rebuild
after the load. **No regex splitting of DDL** — that is explicitly forbidden.

Measured: index-after-load costs 45 s, incremental maintenance costs 6 s. We take the slower path
deliberately, because it guarantees centroids trained on the real distribution rather than on zero
rows, and 45 s is noise against cluster creation.

In [ ]:
with phase("03_schema"):
    schema = subprocess.run(["gcloud","storage","cat",f"{GCS}/schema.sql"],
                            capture_output=True, text=True).stdout
    with connect(DB_NAME) as c:
        c.execute(schema)
        c.execute("DROP INDEX IF EXISTS players_profile_embedding_scann_idx")
        c.execute("DROP INDEX IF EXISTS clubs_profile_embedding_scann_idx")
    print("    schema applied, ScaNN indexes dropped")

## Phase 3 — pass 1, the eight relational tables

Two things that bit us, both guarded here:

- The manifest's `column_order` is **DDL-derived**: for `players` and `clubs` it includes
  `profile_text` / `profile_embedding`, which the pass-1 CSVs do not carry. Subtract them.
- **Preflight the field count.** A list that is *longer* than the file errors loudly. A list the
  same length in a different order loads *silently* into the wrong columns, and the first symptom is
  a lab task returning nonsense on stage.

In [ ]:
PASS2_COLS = {"profile_text","profile_embedding"}
ORDER = ["competitions","clubs","players","games",
         "appearances","game_events","player_valuations","transfers"]

manifest = json.loads(subprocess.run(["gcloud","storage","cat",f"{GCS}/manifest.json"],
                                     capture_output=True, text=True).stdout)
sf = manifest.get("staged_files")
items = sf.items() if isinstance(sf, dict) else [(f.get("name"), f) for f in sf]

COLS = {}
for k, v in items:
    if not isinstance(v, dict):
        continue
    tbl = str(k).split("/")[-1].replace(".csv.gz","").replace(".csv","")
    co = v.get("column_order")
    if co:
        COLS[tbl] = [c for c in co if c not in PASS2_COLS]

for t in ORDER:
    print(f"  {t:20s} {len(COLS.get(t, []))} columns")
assert all(t in COLS for t in ORDER), "manifest missing a table -- never guess column order"

In [ ]:
import csv
with phase("04_preflight"):
    for t in ORDER:
        raw = subprocess.run(["gcloud","storage","cat",f"{GCS}/{t}.csv.gz"],
                             capture_output=True)
        head = gzip.decompress(raw.stdout)[:200_000].decode("utf-8", "replace")
        n_file = len(next(csv.reader(io.StringIO(head))))
        n_cols = len(COLS[t])
        assert n_cols == n_file, f"{t}: list={n_cols} file={n_file} -- REFUSING to load"
        print(f"    {t:20s} {n_cols} columns OK")

In [ ]:
with phase("05_pass1"):
    with connect(DB_NAME) as c:
        for t in ORDER:
            t0 = time.time()
            raw = subprocess.run(["gcloud","storage","cat",f"{GCS}/{t}.csv.gz"],
                                 capture_output=True).stdout
            data = gzip.decompress(raw)
            cols = ",".join(COLS[t])
            with c.cursor().copy(f"COPY {t} ({cols}) FROM STDIN WITH (FORMAT csv)") as cp:
                cp.write(data)
            n = c.execute(f"SELECT count(*) FROM {t}").fetchone()[0]
            print(f"    {t:20s} {n:>8,} rows   {time.time()-t0:5.1f} s")

## Phase 4 — pass 2, profiles

`load_profiles.sql` streams both gzipped CSVs from GCS itself and is wrapped in a transaction, so an
assertion failure rolls back rather than half-loading.

⚠️ Its assertion covers **players only** — `n_clubs` is computed, printed, and never checked. A
zero-row clubs load passes silently. We add the missing check.

⚠️ It uses `\copy ... FROM PROGRAM`, a psql meta-command. From a notebook we re-implement the same
two `UPDATE`s directly rather than shelling out to psql.

In [ ]:
with phase("06_pass2"):
    with connect(DB_NAME) as c:
        for tbl, key, fname in [("players","player_id","players_profiles.csv.gz"),
                                ("clubs","club_id","clubs_profiles.csv.gz")]:
            t0 = time.time()
            raw = gzip.decompress(subprocess.run(
                ["gcloud","storage","cat",f"{GCS}/{fname}"], capture_output=True).stdout)
            c.execute(f"CREATE TEMP TABLE _t ({key} INTEGER, profile_text TEXT, "
                      f"profile_embedding VECTOR(3072))")
            with c.cursor().copy("COPY _t FROM STDIN WITH (FORMAT csv)") as cp:
                cp.write(raw)
            c.execute(f'''UPDATE {tbl} x SET profile_text = t.profile_text,
                          profile_embedding = t.profile_embedding
                          FROM _t t WHERE x.{key} = t.{key}''')
            c.execute("DROP TABLE _t")
            n = c.execute(f"SELECT count(*) FROM {tbl} "
                          f"WHERE profile_embedding IS NOT NULL").fetchone()[0]
            print(f"    {tbl:10s} {n:>8,} profiles  {time.time()-t0:5.1f} s")

        p = c.execute("SELECT count(*) FROM players WHERE profile_embedding IS NOT NULL").fetchone()[0]
        k = c.execute("SELECT count(*) FROM clubs   WHERE profile_embedding IS NOT NULL").fetchone()[0]
        assert p >= 13304, f"player profile load short: {p}"
        assert k >= 790,  f"club profile load short: {k}"   # the check load_profiles.sql lacks

## Phase 5 — ScaNN, after the load

`num_leaves` ≈ √rows: 116 for 13,439 players, 28 for 796 clubs. Measured at 27 s and 18 s from a VM.

**This is the phase most worth moving into student view.** ScaNN is a featured Google product and
right now it is buried in a shell script nobody reads.

In [ ]:
with phase("07_scann"):
    with connect(DB_NAME) as c:
        c.execute("SET maintenance_work_mem='4GB'")
        for tbl, leaves in [("players",116), ("clubs",28)]:
            t0 = time.time()
            c.execute(f'''CREATE INDEX {tbl}_profile_embedding_scann_idx ON {tbl}
                          USING scann (profile_embedding cosine)
                          WITH (num_leaves={leaves}, quantizer='sq8')''')
            print(f"    {tbl:10s} {time.time()-t0:5.1f} s")
        show_notices(c)
        c.execute("ANALYZE players"); c.execute("ANALYZE clubs")

## Does a notebook show NOTICE output?

The open question from the Studio discussion. If `psycopg` surfaces these, Task 3 gets its
instrumentation — `k1`, `b`, document count, average length — for free.

In [ ]:
with phase("08_bm25_notice_check"):
    with connect(DB_NAME) as c:
        c.execute("DROP INDEX IF EXISTS players_profile_text_bm25_idx")
        c.execute('''CREATE INDEX players_profile_text_bm25_idx ON players
                     USING bm25 (profile_text) WITH (text_config = 'english')''')
        print("  NOTICES:")
        show_notices(c)
        print("  ^ expect: 13439 documents, avg_length=161.68")
        c.execute("DROP INDEX players_profile_text_bm25_idx")   # Task 3 builds it, not us

## Result

In [ ]:
with connect(DB_NAME) as c:
    c.execute('''CREATE TABLE IF NOT EXISTS provisioning_status (
                   finished_at timestamptz PRIMARY KEY DEFAULT now(),
                   players bigint, clubs bigint, appearances bigint)''')
    c.execute('''INSERT INTO provisioning_status (players, clubs, appearances)
                 SELECT (SELECT count(*) FROM players WHERE profile_embedding IS NOT NULL),
                        (SELECT count(*) FROM clubs   WHERE profile_embedding IS NOT NULL),
                        (SELECT count(*) FROM appearances)''')
    print(c.execute("SELECT * FROM provisioning_status").fetchall())

print("\n=========== PHASE TIMINGS ===========")
for k, v in TIMINGS.items():
    print(f"  {k:28s} {v:7.1f} s")
print(f"  {'TOTAL':28s} {sum(TIMINGS.values()):7.1f} s")
print("=====================================")
print("VM baseline: pass1 82s + pass2 14s + scann 45s = ~141 s")
print("Add Colab runtime cold start (measure separately) for the real Task 1 cost.")